In [ ]:
import os
from pypdf import PdfReader
folder = "course"
documents = []
for filename in os.listdir(folder):
    if filename.endswith(".pdf"):
        file_path = os.path.join(folder, filename)
        reader = PdfReader(file_path)
        for page_number, page in enumerate(reader.pages):
            text = page.extract_text()
            documents.append({
                "Source": filename,
                "page_number": page_number + 1,
                "text": text
                })


In [ ]:
documents


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
)


In [ ]:
chunks = []
for document in documents:
    page_chunks = text_splitter.split_text(document["text"])
    for chunk in page_chunks:
        chunks.append({
            "Source": document["Source"],
            "page_number": document["page_number"],
            "text": chunk
        })


In [ ]:
chunks


In [30]:
from sentence_transformers import SentenceTransformer
text = [chunk["text"] for chunk in chunks]
model = SentenceTransformer('all-MiniLM-L6-v2')
embedder = model.encode(text)
metadata = [{
    "Source": chunk["Source"], "page_number": chunk["page_number"]
    } for chunk in chunks]


In [38]:
import chromadb
client = chromadb.PersistentClient(
    path="assistant_db"
)
try:
    client.delete_collection("my_three")
except:
    pass
collection = client.get_or_create_collection(
    name="my_three"
)
collection.add(
    ids=[str(i) for i in range(len(embedder))],
    embeddings=embedder.tolist(),
    metadatas=metadata,
    documents=text
)


In [ ]:
query = "What is syllabus in module 3"
q_embedding = model.encode([query])
results = collection.query(
    query_embeddings=q_embedding.tolist(),
    n_results=3
)
results


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
load_dotenv()
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.2, 
    max_output_tokens=1024
    )


Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [ ]:
# res = results['documents'][0]
# content = '\n\n'.join(res)
# prompt = f"""
# You are a question-answering assistant.

# Answer the question using only the provided context.

# If the answer is not present in the context, say:
# "I could not find that information in the provided source."

# Context:
# {content}

# Question:
# {query}

# Answer:
# """


In [ ]:
# response = llm.invoke(prompt)
# print(response.content)


In [36]:
def ask_RAG(que):
    q_embedding = model.encode([que])
    results = collection.query(
        query_embeddings=q_embedding.tolist(),
        n_results=3
    )
    res = results['documents'][0]
    context = '\n\n'.join(res)
    prompt = f"""
    You are a question-answering assistant.

    Answer the question using only the provided context.

    If the answer is not present in the context, say:
    "I could not find that information in the provided source."

    Context:
    {context}

    Question:
    {que}

    Answer:
    """
    response = llm.invoke(prompt)
    return response.content


In [ ]:
import gradio as gr
def answer_question(question,history):
    return ask_RAG(question)
gr.ChatInterface(
    fn=answer_question,
    title="Course Assistant",
).launch(inbrowser=True)


* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


c:\Users\Mohammed Baqar\AppData\Local\Programs\Python\Python313\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Users\Mohammed Baqar\AppData\Local\Programs\Python\Python313\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Users\Mohammed Baqar\AppData\Local\Programs\Python\Python313\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Users\Mohammed Baqar\AppData\Local\Programs\Python\Python313\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_U